In [10]:
import os
import numpy as np
import pandas as pd
from shutil import copyfile

In [2]:
stats_df = pd.read_csv("./articles_genders_stats.csv")
stats_df

,file,num_man_paragraphs,num_woman_paragraphs,num_equal_paragraphs,num_genderless_paragraphs,num_tokens,num_paragraphs
0,news_10-get-ready-for-a-november-to-remember-i...,24,11,0,9,3000,44
1,news_10-june-schedule-jam-packed-goodness.txt,30,8,0,10,2761,48
2,news_10-massive-fights-schedule-may.txt,22,16,0,9,2885,47
3,news_10-memorable-moments-down-under.txt,27,4,0,15,2769,46
4,news_10-middleweights-biggest-championship-mom...,57,0,0,10,3273,67
...,...,...,...,...,...,...,...
876,news_zhang-mingyang-embracing-spotlight-ufc-sh...,9,0,1,3,617,13
877,news_zhang-mingyang-journey-main-event-light-h...,8,0,0,0,728,8
878,news_zhang-weili-big-challenge-apple-vechain-u...,0,19,0,5,882,24
879,news_zhang-weili-builds-her-case-goat-status.txt,0,0,0,1,0,1


In [3]:
stats_df["prop_man_paragraphs"] = stats_df["num_man_paragraphs"] / stats_df["num_paragraphs"]
stats_df["prop_woman_paragraphs"] = stats_df["num_woman_paragraphs"] / stats_df["num_paragraphs"]
stats_df["prop_equal_paragraphs"] = stats_df["num_equal_paragraphs"] / stats_df["num_paragraphs"]
stats_df["prop_genderless_paragraphs"] = stats_df["num_genderless_paragraphs"] / stats_df["num_paragraphs"]

stats_df

,file,num_man_paragraphs,num_woman_paragraphs,num_equal_paragraphs,num_genderless_paragraphs,num_tokens,num_paragraphs,prop_man_paragraphs,prop_woman_paragraphs,prop_equal_paragraphs,prop_genderless_paragraphs
0,news_10-get-ready-for-a-november-to-remember-i...,24,11,0,9,3000,44,0.545455,0.250000,0.000000,0.204545
1,news_10-june-schedule-jam-packed-goodness.txt,30,8,0,10,2761,48,0.625000,0.166667,0.000000,0.208333
2,news_10-massive-fights-schedule-may.txt,22,16,0,9,2885,47,0.468085,0.340426,0.000000,0.191489
3,news_10-memorable-moments-down-under.txt,27,4,0,15,2769,46,0.586957,0.086957,0.000000,0.326087
4,news_10-middleweights-biggest-championship-mom...,57,0,0,10,3273,67,0.850746,0.000000,0.000000,0.149254
...,...,...,...,...,...,...,...,...,...,...,...
876,news_zhang-mingyang-embracing-spotlight-ufc-sh...,9,0,1,3,617,13,0.692308,0.000000,0.076923,0.230769
877,news_zhang-mingyang-journey-main-event-light-h...,8,0,0,0,728,8,1.000000,0.000000,0.000000,0.000000
878,news_zhang-weili-big-challenge-apple-vechain-u...,0,19,0,5,882,24,0.000000,0.791667,0.000000,0.208333
879,news_zhang-weili-builds-her-case-goat-status.txt,0,0,0,1,0,1,0.000000,0.000000,0.000000,1.000000


We want to sample 5% of the corpus. 2.5% of the articles will be mostly about woman, while the other 2.5% will have an equal balance of paragraphs. 

In [4]:
half_sample_corpus_size = int(stats_df.shape[0] * 0.025)
half_sample_corpus_size

22

In [5]:
PROP_WOMAN_PARAGRAPHS_THRESHOLD = 0.9

In [6]:
mostly_woman_df = stats_df[stats_df["prop_woman_paragraphs"] >= PROP_WOMAN_PARAGRAPHS_THRESHOLD]\
                    .sort_values("prop_woman_paragraphs", ascending=False)

# Get the most "average" articles
mostly_woman_average_tokens = mostly_woman_df["num_tokens"].median()
mostly_woman_df_sorted = mostly_woman_df.sort_values(
    by="num_tokens", 
    key=lambda num_tokens: abs(num_tokens - mostly_woman_average_tokens)
)

mostly_woman_to_save = mostly_woman_df_sorted.head(half_sample_corpus_size)
mostly_woman_to_save

,file,num_man_paragraphs,num_woman_paragraphs,num_equal_paragraphs,num_genderless_paragraphs,num_tokens,num_paragraphs,prop_man_paragraphs,prop_woman_paragraphs,prop_equal_paragraphs,prop_genderless_paragraphs
26,news_alexa-grasso-embracing-every-opportunity-...,0,14,0,1,822,15,0.0,0.933333,0.0,0.066667
556,news_matchup-to-watch-vieira-chiasson-ufc-vega...,0,9,0,1,821,10,0.0,0.900000,0.0,0.100000
529,news_manon-fiorot-journey-to-the-title-shot-uf...,0,12,0,1,808,13,0.0,0.923077,0.0,0.076923
42,news_amanda-ribas-confident-and-competitive-uf...,0,13,0,1,839,14,0.0,0.928571,0.0,0.071429
518,news_main-event-spotlight-ufc-fight-night-blan...,0,13,0,0,849,13,0.0,1.000000,0.0,0.000000
309,news_flyweights-in-the-spotlight-in-montreal-u...,0,10,0,0,794,10,0.0,1.000000,0.0,0.000000
350,news_iasmin-lucindo-letting-flow-ufc-313.txt,0,13,0,1,853,14,0.0,0.928571,0.0,0.071429
40,news_amanda-lemos-eyes-prize-noche-ufc.txt,0,21,0,2,789,23,0.0,0.913043,0.0,0.086957
472,news_loopy-godinez-homecoming-ufc-mexico.txt,0,11,0,1,784,12,0.0,0.916667,0.0,0.083333
866,news_womens-flyweight-division-2026-preview.txt,0,13,0,1,875,14,0.0,0.928571,0.0,0.071429


In [7]:
stats_df["entropy"] = -(
    stats_df["prop_man_paragraphs"] * np.log(stats_df["prop_man_paragraphs"] + 1e-12)
    + stats_df["prop_woman_paragraphs"] * np.log(stats_df["prop_woman_paragraphs"] + 1e-12)
    + stats_df["prop_equal_paragraphs"] * np.log(stats_df["prop_equal_paragraphs"] + 1e-12)
    + stats_df["prop_genderless_paragraphs"] * np.log(stats_df["prop_genderless_paragraphs"] + 1e-12)
)

# Max entropy for 4 categories is log(4)
stats_df["entropy_norm"] = stats_df["entropy"] / np.log(4)

stats_df["variance"] = \
                    (stats_df["prop_man_paragraphs"] - 0.25) ** 2 \
                    + (stats_df["prop_woman_paragraphs"] - 0.25) ** 2 \
                    + (stats_df["prop_equal_paragraphs"] - 0.25) ** 2 \
                    + (stats_df["prop_genderless_paragraphs"] - 0.25) ** 2

# Max possible variance for 4 categories is 0.75
stats_df["variance_norm"] = stats_df["variance"] / 0.75

# Balance out the number of tokens to try to make them as equal as possible
stats_df["tokens_balance"] = abs(stats_df["num_tokens"].median() - stats_df["num_tokens"]) / stats_df["num_tokens"].max()

stats_df["balance"] = stats_df["entropy_norm"] \
                    + (1 - stats_df["variance_norm"]) \
                    + stats_df["tokens_balance"] * 0.2 # Don't weight the token balance as much

balanced_to_save = stats_df.sort_values("balance", ascending=False).head(half_sample_corpus_size)
balanced_to_save

,file,num_man_paragraphs,num_woman_paragraphs,num_equal_paragraphs,num_genderless_paragraphs,num_tokens,num_paragraphs,prop_man_paragraphs,prop_woman_paragraphs,prop_equal_paragraphs,prop_genderless_paragraphs,entropy,entropy_norm,variance,variance_norm,tokens_balance,balance
604,news_monthly-report-may-2025.txt,20,9,2,10,2559,41,0.487805,0.219512,0.048780,0.243902,1.174503,0.847225,0.098007,0.130676,0.176406,1.751830
653,news_prelim-live-results-fight-recaps-highligh...,7,2,1,5,884,15,0.466667,0.133333,0.066667,0.333333,1.171060,0.844741,0.101111,0.134815,0.028362,1.715599
670,news_prelim-results-highlights-winner-intervie...,5,4,1,1,513,11,0.454545,0.363636,0.090909,0.090909,1.162226,0.838369,0.105372,0.140496,0.073716,1.712616
671,news_prelim-results-highlights-winner-intervie...,11,6,1,4,955,22,0.500000,0.272727,0.045455,0.181818,1.151380,0.830545,0.109504,0.146006,0.019682,1.688476
754,news_significant-stats-ufc-312-du-plessis-vs-s...,3,3,0,4,769,10,0.300000,0.300000,0.000000,0.400000,1.088900,0.785475,0.090000,0.120000,0.042421,1.673959
769,news_submissions-2025-ufccom-awards.txt,13,3,2,6,1579,24,0.541667,0.125000,0.083333,0.250000,1.145678,0.826432,0.128472,0.171296,0.056601,1.666456
682,news_prelim-results-ufc-fight-night-royval-vs-...,3,2,0,2,416,7,0.428571,0.285714,0.000000,0.285714,1.078992,0.778328,0.096939,0.129252,0.085575,1.666192
810,news_ufc-315-best-bets-draftkings-sportsbook.txt,6,4,0,4,669,14,0.428571,0.285714,0.000000,0.285714,1.078992,0.778328,0.096939,0.129252,0.054645,1.660006
603,news_monthly-report-march-2025.txt,14,10,0,8,1859,32,0.437500,0.312500,0.000000,0.250000,1.071730,0.773090,0.101562,0.135417,0.090831,1.655839
601,news_monthly-report-january-2025.txt,12,10,0,7,1540,29,0.413793,0.344828,0.000000,0.241379,1.075361,0.775709,0.098395,0.131193,0.051834,1.654883


In [8]:
files_to_save = set(mostly_woman_to_save["file"].to_list() + balanced_to_save["file"].to_list())
len(files_to_save)

44

In [11]:
os.makedirs("./sample_corpus", exist_ok=True)

In [13]:
for file in files_to_save:
    copyfile(f"./articles/{file}", f"./sample_corpus/{file}")